# Comprehensive Welding Dataset Analysis for ML-Driven Inverse Design

This notebook provides comprehensive analysis of the welding dataset for inverse design applications.

## Dataset Overview
The dataset contains three main components:
1. **Input Parameters**: Welding process parameters we can control
2. **Characterization Metrics**: Quality measurements immediately after welding
3. **Performance Metrics**: Long-term performance under extreme conditions

## Key Features
- Physics-based relationships between parameters and outcomes
- Realistic material property variations
- Multiple welding techniques (USW, Laser, Resistance)
- Comprehensive performance metrics for extreme-temperature cycling

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set plotting parameters
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
# Load the dataset
dataset = pd.read_csv('welding_dataset.csv')
print(f"Dataset shape: {dataset.shape}")
print(f"Memory usage: {dataset.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
dataset.head()

## 1. Dataset Overview and Basic Statistics

In [ ]:
# Basic dataset information
print("Dataset Information:")
print(f"Total samples: {len(dataset)}")
print(f"Total features: {len(dataset.columns)}")
print(f"Missing values: {dataset.isnull().sum().sum()}")
print(f"Duplicate rows: {dataset.duplicated().sum()}")

# Data types
print("\nData Types:")
print(dataset.dtypes.value_counts())

# Numerical columns summary
numerical_cols = dataset.select_dtypes(include=[np.number]).columns
print(f"\nNumerical columns: {len(numerical_cols)}")
print(f"Categorical columns: {len(dataset.columns) - len(numerical_cols)}")

In [ ]:
# Categorical variables distribution
categorical_cols = ['anode_material', 'cathode_material', 'surface_finish', 'welding_technique']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    if col in dataset.columns:
        value_counts = dataset[col].value_counts()
        axes[i].pie(value_counts.values, labels=value_counts.index, autopct='%1.1f%%')
        axes[i].set_title(f'Distribution of {col}')

plt.tight_layout()
plt.show()

# Print exact counts
for col in categorical_cols:
    if col in dataset.columns:
        print(f"\n{col}:")
        print(dataset[col].value_counts())

## 2. Input Parameters Analysis

In [ ]:
# Input parameters (what we can control)
input_params = [
    'tab_thickness_um', 'power_W', 'amplitude_um', 'force_N', 'time_s',
    'speed_mm_s', 'pulse_frequency_Hz', 'pulse_energy_J', 'current_A', 'pre_heat_temp_C'
]

# Filter to existing columns
input_params = [col for col in input_params if col in dataset.columns]

print("Input Parameters Summary:")
print(dataset[input_params].describe())

# Visualize input parameter distributions
n_params = len(input_params)
n_cols = 3
n_rows = (n_params + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6*n_rows))
axes = axes.flatten() if n_rows > 1 else [axes] if n_rows == 1 else axes

for i, param in enumerate(input_params):
    if i < len(axes):
        # Remove NaN values for plotting
        data_clean = dataset[param].dropna()
        if len(data_clean) > 0:
            axes[i].hist(data_clean, bins=50, alpha=0.7, edgecolor='black')
            axes[i].set_title(f'Distribution of {param}')
            axes[i].set_xlabel(param)
            axes[i].set_ylabel('Frequency')
            
            # Add statistics
            mean_val = data_clean.mean()
            std_val = data_clean.std()
            axes[i].axvline(mean_val, color='red', linestyle='--', label=f'Mean: {mean_val:.2f}')
            axes[i].axvline(mean_val + std_val, color='orange', linestyle='--', label=f'+1σ')
            axes[i].axvline(mean_val - std_val, color='orange', linestyle='--', label=f'-1σ')
            axes[i].legend()

# Hide empty subplots
for i in range(len(input_params), len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.show()

## 3. Characterization Metrics Analysis (Forward Problem Outputs)

In [ ]:
# Characterization metrics (immediate quality measurements)
char_metrics = [
    'heat_input_J', 'contact_resistance_Ohm', 'weld_strength_MPa',
    'heat_affected_zone_mm', 'porosity_percent', 'weld_width_mm',
    'penetration_depth_mm', 'microhardness_HV', 'total_resistance_Ohm',
    'thermal_resistance_K_W'
]

# Filter to existing columns
char_metrics = [col for col in char_metrics if col in dataset.columns]

print("Characterization Metrics Summary:")
print(dataset[char_metrics].describe())

# Visualize characterization metrics
n_metrics = len(char_metrics)
n_cols = 3
n_rows = (n_metrics + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6*n_rows))
axes = axes.flatten() if n_rows > 1 else [axes] if n_rows == 1 else axes

for i, metric in enumerate(char_metrics):
    if i < len(axes):
        data_clean = dataset[metric].dropna()
        if len(data_clean) > 0:
            axes[i].hist(data_clean, bins=50, alpha=0.7, edgecolor='black')
            axes[i].set_title(f'Distribution of {metric}')
            axes[i].set_xlabel(metric)
            axes[i].set_ylabel('Frequency')
            
            # Add statistics
            mean_val = data_clean.mean()
            std_val = data_clean.std()
            axes[i].axvline(mean_val, color='red', linestyle='--', label=f'Mean: {mean_val:.2f}')
            axes[i].axvline(mean_val + std_val, color='orange', linestyle='--', label=f'+1σ')
            axes[i].axvline(mean_val - std_val, color='orange', linestyle='--', label=f'-1σ')
            axes[i].legend()

# Hide empty subplots
for i in range(len(char_metrics), len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.show()

## 4. Performance Metrics Analysis (Inverse Design Targets)

In [ ]:
# Performance metrics (long-term performance under extreme conditions)
perf_metrics = [
    'thermal_cycles_to_failure', 'high_temp_strength_MPa', 'fatigue_cycles_1e6',
    'resistance_after_cycling_Ohm', 'thermal_performance_W_mK', 'creep_resistance_MPa',
    'interfacial_stability', 'thermal_expansion_mismatch_1e6_K'
]

# Filter to existing columns
perf_metrics = [col for col in perf_metrics if col in dataset.columns]

print("Performance Metrics Summary:")
print(dataset[perf_metrics].describe())

# Visualize performance metrics
n_metrics = len(perf_metrics)
n_cols = 3
n_rows = (n_metrics + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6*n_rows))
axes = axes.flatten() if n_rows > 1 else [axes] if n_rows == 1 else axes

for i, metric in enumerate(perf_metrics):
    if i < len(axes):
        data_clean = dataset[metric].dropna()
        if len(data_clean) > 0:
            axes[i].hist(data_clean, bins=50, alpha=0.7, edgecolor='black')
            axes[i].set_title(f'Distribution of {metric}')
            axes[i].set_xlabel(metric)
            axes[i].set_ylabel('Frequency')
            
            # Add statistics
            mean_val = data_clean.mean()
            std_val = data_clean.std()
            axes[i].axvline(mean_val, color='red', linestyle='--', label=f'Mean: {mean_val:.2f}')
            axes[i].axvline(mean_val + std_val, color='orange', linestyle='--', label=f'+1σ')
            axes[i].axvline(mean_val - std_val, color='orange', linestyle='--', label=f'-1σ')
            axes[i].legend()

# Hide empty subplots
for i in range(len(perf_metrics), len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
# Calculate correlation matrix for numerical variables
numerical_cols = dataset.select_dtypes(include=[np.number]).columns
corr_matrix = dataset[numerical_cols].corr()

# Plot correlation heatmap
plt.figure(figsize=(20, 16))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
           square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, fmt='.2f')
plt.title('Correlation Matrix of Welding Parameters and Performance Metrics', 
         fontsize=16, pad=20)
plt.tight_layout()
plt.show()

# Find high correlations
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_val = corr_matrix.iloc[i, j]
        if abs(corr_val) > 0.7:
            high_corr.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_val))

print("High Correlations (|r| > 0.7):")
for var1, var2, corr in sorted(high_corr, key=lambda x: abs(x[2]), reverse=True):
    print(f"{var1} <-> {var2}: {corr:.3f}")

## 6. Performance by Welding Technique

In [ ]:
# Compare performance metrics by welding technique
if 'welding_technique' in dataset.columns:
    # Key performance metrics
    key_metrics = ['thermal_cycles_to_failure', 'weld_strength_MPa', 'porosity_percent', 'interfacial_stability']
    key_metrics = [col for col in key_metrics if col in dataset.columns]
    
    n_metrics = len(key_metrics)
    n_cols = 2
    n_rows = (n_metrics + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 6*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_rows == 1 else axes
    
    for i, metric in enumerate(key_metrics):
        if i < len(axes):
            sns.boxplot(data=dataset, x='welding_technique', y=metric, ax=axes[i])
            axes[i].set_title(f'{metric} by Welding Technique')
            axes[i].tick_params(axis='x', rotation=45)
    
    # Hide empty subplots
    for i in range(len(key_metrics), len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    # Statistical comparison
    print("Statistical Comparison by Welding Technique:")
    for metric in key_metrics:
        print(f"\n{metric}:")
        technique_stats = dataset.groupby('welding_technique')[metric].agg(['mean', 'std', 'min', 'max'])
        print(technique_stats)

## 7. Material Combination Analysis

In [ ]:
# Analyze material combinations
if 'anode_material' in dataset.columns and 'cathode_material' in dataset.columns:
    # Create material combination column
    dataset['material_combination'] = (
        dataset['anode_material'] + '-' + dataset['cathode_material']
    )
    
    # Count combinations
    combo_counts = dataset['material_combination'].value_counts()
    print("Material Combinations:")
    print(combo_counts)
    
    # Performance by material combination
    key_metrics = ['thermal_cycles_to_failure', 'weld_strength_MPa']
    key_metrics = [col for col in key_metrics if col in dataset.columns]
    
    fig, axes = plt.subplots(1, len(key_metrics), figsize=(15, 6))
    if len(key_metrics) == 1:
        axes = [axes]
    
    for i, metric in enumerate(key_metrics):
        # Get top 10 combinations
        top_combos = combo_counts.head(10).index
        subset = dataset[dataset['material_combination'].isin(top_combos)]
        
        sns.boxplot(data=subset, x='material_combination', y=metric, ax=axes[i])
        axes[i].set_title(f'{metric} by Material Combination')
        axes[i].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # Best performing combinations
    print("\nBest Performing Material Combinations:")
    for metric in key_metrics:
        print(f"\n{metric}:")
        best_combos = dataset.groupby('material_combination')[metric].mean().sort_values(ascending=False).head(5)
        print(best_combos)

## 8. Process Parameter Optimization Analysis

In [ ]:
# Analyze relationship between process parameters and performance
process_params = ['power_W', 'force_N', 'time_s', 'amplitude_um', 'speed_mm_s']
process_params = [col for col in process_params if col in dataset.columns]
target_metric = 'thermal_cycles_to_failure'

if target_metric in dataset.columns:
    n_params = len(process_params)
    n_cols = 3
    n_rows = (n_params + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_rows == 1 else axes
    
    for i, param in enumerate(process_params):
        if i < len(axes):
            # Remove NaN values
            valid_data = dataset.dropna(subset=[param, target_metric])
            
            if len(valid_data) > 10:
                # Scatter plot
                axes[i].scatter(valid_data[param], valid_data[target_metric], 
                              alpha=0.6, s=20)
                axes[i].set_xlabel(param)
                axes[i].set_ylabel(target_metric)
                axes[i].set_title(f'{param} vs {target_metric}')
                
                # Add trend line
                z = np.polyfit(valid_data[param], valid_data[target_metric], 1)
                p = np.poly1d(z)
                axes[i].plot(valid_data[param], p(valid_data[param]), 
                           "r--", alpha=0.8, label=f'Slope: {z[0]:.2f}')
                axes[i].legend()
    
    # Hide empty subplots
    for i in range(len(process_params), len(axes)):
        axes[i].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    # Calculate correlations
    print("Correlations with Thermal Cycles to Failure:")
    for param in process_params:
        if param in dataset.columns:
            corr = dataset[param].corr(dataset[target_metric])
            print(f"{param}: {corr:.3f}")

## 9. Interactive 3D Visualization

In [ ]:
# Create interactive 3D scatter plot
if all(col in dataset.columns for col in ['power_W', 'force_N', 'thermal_cycles_to_failure']):
    fig = px.scatter_3d(
        dataset, 
        x='power_W', 
        y='force_N', 
        z='thermal_cycles_to_failure',
        color='welding_technique',
        size='weld_strength_MPa',
        hover_data=['anode_material', 'cathode_material', 'surface_finish'],
        title='3D Visualization: Power vs Force vs Thermal Cycles to Failure',
        labels={
            'power_W': 'Power (W)',
            'force_N': 'Force (N)',
            'thermal_cycles_to_failure': 'Thermal Cycles to Failure'
        }
    )
    
    fig.update_layout(
        scene=dict(
            xaxis_title='Power (W)',
            yaxis_title='Force (N)',
            zaxis_title='Thermal Cycles to Failure'
        )
    )
    
    fig.show()
    
    # Save as HTML
    fig.write_html('interactive_3d_plot.html')
    print("Interactive 3D plot saved as 'interactive_3d_plot.html'")

## 10. Machine Learning Model Training and Inverse Design

In [ ]:
# Import ML libraries
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# Prepare data for ML
def prepare_ml_data(df):
    """Prepare data for machine learning"""
    # Select features
    feature_cols = [
        'tab_thickness_um', 'power_W', 'amplitude_um', 'force_N', 'time_s',
        'speed_mm_s', 'pulse_frequency_Hz', 'pulse_energy_J', 'current_A', 'pre_heat_temp_C'
    ]
    
    # Select target
    target_col = 'thermal_cycles_to_failure'
    
    # Filter existing columns
    feature_cols = [col for col in feature_cols if col in df.columns]
    
    if target_col not in df.columns:
        print(f"Target column {target_col} not found")
        return None, None
    
    # Prepare features
    X = df[feature_cols].copy()
    y = df[target_col].copy()
    
    # Handle missing values
    X = X.fillna(X.median())
    y = y.fillna(y.median())
    
    return X, y

# Prepare data
X, y = prepare_ml_data(dataset)

if X is not None and y is not None:
    print(f"Features shape: {X.shape}")
    print(f"Target shape: {y.shape}")
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    # Train model
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    print(f"\nModel Performance:")
    print(f"RMSE: {rmse:.2f}")
    print(f"R²: {r2:.3f}")
    
    # Feature importance
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(f"\nFeature Importance:")
    print(feature_importance)
    
    # Plot feature importance
    plt.figure(figsize=(10, 6))
    sns.barplot(data=feature_importance, x='importance', y='feature')
    plt.title('Feature Importance for Thermal Cycles to Failure Prediction')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()
    
    # Save model
    joblib.dump(model, 'thermal_cycles_model.pkl')
    print("\nModel saved as 'thermal_cycles_model.pkl'")

## 11. Summary and Conclusions

In [ ]:
# Generate summary statistics
print("=" * 60)
print("WELDING DATASET ANALYSIS SUMMARY")
print("=" * 60)

print(f"\nDataset Overview:")
print(f"- Total samples: {len(dataset):,}")
print(f"- Total features: {len(dataset.columns)}")
print(f"- Memory usage: {dataset.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\nInput Parameters:")
input_params = [col for col in dataset.columns if col in [
    'tab_thickness_um', 'power_W', 'amplitude_um', 'force_N', 'time_s',
    'speed_mm_s', 'pulse_frequency_Hz', 'pulse_energy_J', 'current_A', 'pre_heat_temp_C'
]]
print(f"- Count: {len(input_params)}")
print(f"- Parameters: {', '.join(input_params)}")

print(f"\nCharacterization Metrics:")
char_metrics = [col for col in dataset.columns if col in [
    'heat_input_J', 'contact_resistance_Ohm', 'weld_strength_MPa',
    'heat_affected_zone_mm', 'porosity_percent', 'weld_width_mm',
    'penetration_depth_mm', 'microhardness_HV', 'total_resistance_Ohm',
    'thermal_resistance_K_W'
]]
print(f"- Count: {len(char_metrics)}")
print(f"- Metrics: {', '.join(char_metrics)}")

print(f"\nPerformance Metrics:")
perf_metrics = [col for col in dataset.columns if col in [
    'thermal_cycles_to_failure', 'high_temp_strength_MPa', 'fatigue_cycles_1e6',
    'resistance_after_cycling_Ohm', 'thermal_performance_W_mK', 'creep_resistance_MPa',
    'interfacial_stability', 'thermal_expansion_mismatch_1e6_K'
]]
print(f"- Count: {len(perf_metrics)}")
print(f"- Metrics: {', '.join(perf_metrics)}")

print(f"\nWelding Techniques:")
if 'welding_technique' in dataset.columns:
    technique_counts = dataset['welding_technique'].value_counts()
    for technique, count in technique_counts.items():
        print(f"- {technique}: {count:,} samples ({count/len(dataset)*100:.1f}%)")

print(f"\nMaterial Combinations:")
if 'anode_material' in dataset.columns and 'cathode_material' in dataset.columns:
    material_combos = dataset.groupby(['anode_material', 'cathode_material']).size().sort_values(ascending=False)
    print(f"- Total unique combinations: {len(material_combos)}")
    print(f"- Top 5 combinations:")
    for (anode, cathode), count in material_combos.head().items():
        print(f"  {anode}-{cathode}: {count:,} samples")

print(f"\nData Quality:")
print(f"- Missing values: {dataset.isnull().sum().sum():,}")
print(f"- Duplicate rows: {dataset.duplicated().sum():,}")
print(f"- Data completeness: {(1 - dataset.isnull().sum().sum() / (len(dataset) * len(dataset.columns))) * 100:.1f}%")

print(f"\n" + "=" * 60)
print("ANALYSIS COMPLETE")
print("=" * 60)